# 🦉 BadBoerdi — Komplett-Stack in Google Colab

Startet **Backend (FastAPI)**, **Frontend (Angular)** und **Studio (Next.js)** in einer Colab-VM und tunnelt alle drei Dienste über **cloudflared** (kein Account nötig).

**Voraussetzungen:**
- Ein OpenAI API Key
- Eine Colab-Sitzung (CPU reicht; T4-GPU bringt nichts)

**Caveats:**
- Colab trennt nach ~90 min Idle und nach max. 12 h
- Sessions liegen in `/content/badboerdi/badboerdi.db`, nach Disconnect weg
- Studio kann zwar Dateien editieren (Colab-FS ist beschreibbar), Aenderungen sind aber genauso ephemer

## 1. Repository klonen

In [ ]:
import os

REPO_URL = "https://github.com/janschachtschabel/badboerdi-chatframework.git"
BRANCH   = "main"

os.chdir('/content')
if os.path.isdir('badboerdi') and not os.path.isdir('badboerdi/.git'):
    import shutil; shutil.rmtree('badboerdi')
    print('Defektes Verzeichnis entfernt')

if not os.path.isdir('badboerdi'):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} badboerdi
else:
    !cd badboerdi && git pull --ff-only

%cd /content/badboerdi
!ls

## 2. System-Abhängigkeiten (Node 20 + cloudflared)

Colab bringt Python 3.11 mit, aber kein modernes Node. Wir installieren Node 20 und das `cloudflared`-Binary.

In [ ]:
%%bash
set -e
if ! command -v node >/dev/null || [ "$(node -v | cut -d. -f1)" != "v20" ]; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1
  apt-get install -y nodejs >/dev/null 2>&1
fi
node -v && npm -v
if ! command -v cloudflared >/dev/null; then
  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
  chmod +x /usr/local/bin/cloudflared
fi
cloudflared --version

## 3. Python-Abhängigkeiten (Backend)

In [ ]:
# Colab hat vorinstallierte Pakete (google-adk, gradio, etc.) deren
# Versionen mit unseren kollidieren. Die Warnungen betreffen NUR
# diese Colab-Pakete und sind harmlos — badboerdi nutzt sie nicht.
!pip install -q -r /content/badboerdi/backend/requirements.txt 2>&1 | \
    grep -v "already satisfied" | grep -v "dependency resolver" | \
    grep -v "requires.*but you have" | grep -v "incompatible" | tail -3
print("Backend-Deps OK")

## 4. NPM-Abhängigkeiten (Frontend + Studio)

In [ ]:
%%bash
cd /content/badboerdi/frontend && npm install --silent
cd /content/badboerdi/studio && npm install --silent

## 5. OpenAI API Key hinterlegen

Wir nutzen Colabs Secrets-Speicher. Klicke links auf 🔑 "Secrets" → "+ Add new secret" → Name `OPENAI_API_KEY` → Wert eintragen → Notebook-Zugriff erlauben.

In [ ]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# LLM-Provider-Konfiguration (moderne Env-Vars)
os.environ.setdefault('LLM_PROVIDER', 'openai')
os.environ.setdefault('LLM_CHAT_MODEL', 'gpt-4.1-mini')
os.environ.setdefault('LLM_EMBED_MODEL', 'text-embedding-3-small')

# MCP-Server (Default passt normalerweise)
os.environ.setdefault('MCP_SERVER_URL', 'https://wlo-mcp-server.vercel.app/mcp')

# Datenbank
os.environ['DATABASE_PATH'] = '/content/badboerdi/badboerdi.db'

print('OK — Key gesetzt:', bool(os.environ.get('OPENAI_API_KEY')))
print('Provider:', os.environ.get('LLM_PROVIDER'))
print('Chat-Model:', os.environ.get('LLM_CHAT_MODEL'))

## 6. Backend starten (Port 8000, im Hintergrund)

`reload=True` wird fuer Colab deaktiviert (File-Watcher verursacht Probleme in der VM).

In [ ]:
import subprocess, time, os, urllib.request, json

env = os.environ.copy()
# Colab: File-Watcher verursacht Probleme in der VM -> reload deaktivieren
env['BOERDI_UVICORN_RELOAD'] = '0'

backend_log = open('/content/backend.log', 'w')
backend_proc = subprocess.Popen(
    ['python', 'run.py'],
    cwd='/content/badboerdi/backend',
    env=env,
    stdout=backend_log, stderr=subprocess.STDOUT,
)
print('Backend PID:', backend_proc.pid)

# Health-Check — bis zu 180s warten
# (Colab-Kaltstart + RAG-Embeddings-Init koennen 1-2 Min dauern)
ok = False
for i in range(180):
    try:
        with urllib.request.urlopen('http://localhost:8000/api/health', timeout=2) as r:
            print(f'Backend OK nach {i}s:', json.loads(r.read()))
            ok = True
            break
    except Exception:
        time.sleep(1)

if not ok:
    print('[FAIL] Backend kam nicht hoch — siehe /content/backend.log')
    get_ipython().system('tail -80 /content/backend.log')
    raise RuntimeError('Backend start failed — bitte Logs pruefen, nicht weiter ausfuehren')


## 7. Frontend starten (Port 4200, im Hintergrund)

Angular Dev-Server proxied `/api` automatisch auf `localhost:8000` (siehe `proxy.conf.json`). Für Colab erlauben wir explizit den Tunnel-Host.

In [ ]:
import subprocess
frontend_log = open('/content/frontend.log', 'w')
frontend_proc = subprocess.Popen(
    ['npx', 'ng', 'serve', '--host', '0.0.0.0', '--port', '4200',
     '--disable-host-check', '--proxy-config', 'proxy.conf.json'],
    cwd='/content/badboerdi/frontend',
    stdout=frontend_log, stderr=subprocess.STDOUT,
)
print('Frontend PID:', frontend_proc.pid)
import time
for _ in range(120):
    try:
        import urllib.request
        urllib.request.urlopen('http://localhost:4200', timeout=2)
        print('Frontend OK')
        break
    except Exception:
        time.sleep(2)
else:
    print('Frontend kam nicht hoch — siehe /content/frontend.log')
    !tail -80 /content/frontend.log

## 8. Studio starten (Port 3001, im Hintergrund)

In [ ]:
import subprocess, os

studio_env = os.environ.copy()
studio_env['BACKEND_URL'] = 'http://localhost:8000'  # Studio-Proxy → Backend (lokal auf derselben VM)

studio_log = open('/content/studio.log', 'w')
studio_proc = subprocess.Popen(
    ['npx', 'next', 'dev', '-p', '3001', '-H', '0.0.0.0'],
    cwd='/content/badboerdi/studio',
    env=studio_env,
    stdout=studio_log, stderr=subprocess.STDOUT,
)
print('Studio PID:', studio_proc.pid)
import time
for _ in range(60):
    try:
        import urllib.request
        urllib.request.urlopen('http://localhost:3001', timeout=2)
        print('Studio OK')
        break
    except Exception:
        time.sleep(2)
else:
    print('Studio kam nicht hoch — siehe /content/studio.log')
    !tail -80 /content/studio.log

## 9. Cloudflare Tunnels für alle drei Ports

Jeder Quick-Tunnel bekommt eine zufällige `*.trycloudflare.com`-URL. Kein Login, kein Token nötig.

In [ ]:
import subprocess, re, time, os, urllib.request

# Safety-Gate: Backend MUSS erreichbar sein, sonst liefert der Tunnel 502.
print('Pruefe Backend-Erreichbarkeit...')
_backend_up = False
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:8000/api/health', timeout=2)
        _backend_up = True
        break
    except Exception:
        time.sleep(1)
if not _backend_up:
    raise RuntimeError('Backend nicht erreichbar — Tunnel wuerde 502 liefern. Erst Zelle 7 erneut ausfuehren.')
print('Backend erreichbar — Tunnel wird geoeffnet.')

def start_tunnel(port: int, name: str) -> str:
    log_path = f'/content/cf_{name}.log'
    f = open(log_path, 'w')
    subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate',
         '--url', f'http://localhost:{port}'],
        stdout=f, stderr=subprocess.STDOUT,
    )
    pat = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
    for _ in range(60):
        time.sleep(1)
        try:
            with open(log_path) as fr:
                m = pat.search(fr.read())
                if m:
                    return m.group(0)
        except FileNotFoundError:
            pass
    return ''

backend_url  = start_tunnel(8000, 'backend')
frontend_url = start_tunnel(4200, 'frontend')
studio_url   = start_tunnel(3001, 'studio')

print()
print('🦉 BadBoerdi ist live:')
print('  Backend  (FastAPI):', backend_url + '/api/health')
print('  Widget Demo:       ', backend_url + '/widget/')
print('  Frontend (Angular):', frontend_url)
print('  Studio   (Next.js):', studio_url)


## 10. Embedding-Snippet für die Web-Component

Das Backend liefert das Widget über den Backend-Tunnel aus. Damit kannst du `<boerdi-chat>` auf jeder beliebigen Drittseite einbinden.

In [ ]:
snippet = f'''<script src="{backend_url}/widget/boerdi-widget.js" defer></script>
<boerdi-chat
  api-url="{backend_url}"
  page-context='{{"thema":"colab-test"}}'
  position="bottom-right"
  primary-color="#1c4587">
</boerdi-chat>'''
print(snippet)

## 11. Logs live anzeigen (optional)

In [ ]:
!tail -n 30 /content/backend.log

In [ ]:
!tail -n 30 /content/frontend.log

In [ ]:
!tail -n 30 /content/studio.log

## 12. Aufraeumen / Stop

**Diese Zelle NICHT bei "Run all" ausfuehren!** Sie beendet alle laufenden Dienste.

Fuehre sie nur manuell aus, wenn du die Session beenden willst.

In [ ]:
#@title 🛑 Stop — Alle Dienste beenden (nur manuell ausfuehren!)
#@markdown > **Achtung:** Diese Zelle beendet Backend, Frontend, Studio und Tunnels.

# Sicherheitsabfrage: bei "Run all" wird diese Zelle uebersprungen,
# weil die Variable _MANUAL_STOP nicht gesetzt ist.
_MANUAL_STOP = False  # <-- auf True setzen und Zelle manuell ausfuehren

if not _MANUAL_STOP:
    print("⏭️ Uebersprungen. Setze _MANUAL_STOP = True und fuehre die Zelle manuell aus.")
else:
    !pkill -f 'cloudflared'   || true
    !pkill -f 'next dev'      || true
    !pkill -f 'ng serve'      || true
    !pkill -f 'python run.py' || true
    print('Alle Prozesse beendet.')